In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

import gc
import os

Custom defined functions.

In [2]:
from utils import agg_columns, remove_nan_columns

In [3]:
dataset_dir = "dataset"

Since the dataset is large, we restrict to use the first `NUM_ROWS` of each table. Set to `None` to include all data.

In [4]:
NUM_ROWS = 500000

## Feature engineering

### The bureau tables

Load the bureau tables.

In [5]:
bureau_balance_df = pd.read_csv(os.path.join(dataset_dir, "bureau_balance.csv")).iloc[:NUM_ROWS]
bureau_df = pd.read_csv(os.path.join(dataset_dir, "bureau.csv")).iloc[:NUM_ROWS]

Aggregate the *bureau_balance* table and merge it with the *bureau* table.

In [6]:
agg_bureau_balance_df = agg_columns(bureau_balance_df, groupby_columns=["SK_ID_BUREAU"], prefix="BAL").iloc[:NUM_ROWS]
merged_bureau_bureau_balance_df = pd.merge(bureau_df, agg_bureau_balance_df, on="SK_ID_BUREAU", how="left").iloc[:NUM_ROWS]

Free memory.

In [7]:
del bureau_balance_df, bureau_df, agg_bureau_balance_df
gc.collect()

0

Aggregate the merged table and merge it with the main *applications* table.

In [8]:
merged_bureau_df = agg_columns(
    merged_bureau_bureau_balance_df,
    groupby_columns=["SK_ID_CURR"],
    exclude_columns=["SK_ID_BUREAU"],
    prefix="BUREAU"
)

Free memory.

In [9]:
del merged_bureau_bureau_balance_df
gc.collect()

0

In [10]:
merged_bureau_df

,SK_ID_CURR,BUREAU_DAYS_CREDIT_count,BUREAU_DAYS_CREDIT_mean,BUREAU_DAYS_CREDIT_sum,BUREAU_DAYS_CREDIT_min,BUREAU_DAYS_CREDIT_max,BUREAU_CREDIT_DAY_OVERDUE_count,BUREAU_CREDIT_DAY_OVERDUE_mean,BUREAU_CREDIT_DAY_OVERDUE_sum,BUREAU_CREDIT_DAY_OVERDUE_min,...,BUREAU_CREDIT_TYPE_Microloan_count,BUREAU_CREDIT_TYPE_Microloan_count_norm,BUREAU_CREDIT_TYPE_Mobile operator loan_count,BUREAU_CREDIT_TYPE_Mobile operator loan_count_norm,BUREAU_CREDIT_TYPE_Mortgage_count,BUREAU_CREDIT_TYPE_Mortgage_count_norm,BUREAU_CREDIT_TYPE_Real estate loan_count,BUREAU_CREDIT_TYPE_Real estate loan_count_norm,BUREAU_CREDIT_TYPE_Unknown type of loan_count,BUREAU_CREDIT_TYPE_Unknown type of loan_count_norm
0,100001,7,-735.000000,-5145,-1572,-49,7,0.0,0,0,...,0,0.0,0,0.0,0,0.000000,0,0.0,0,0.0
1,100004,2,-867.000000,-1734,-1326,-408,2,0.0,0,0,...,0,0.0,0,0.0,0,0.000000,0,0.0,0,0.0
2,100011,4,-1773.000000,-7092,-2508,-1309,4,0.0,0,0,...,0,0.0,0,0.0,0,0.000000,0,0.0,0,0.0
3,100013,1,-2005.000000,-2005,-2005,-2005,1,0.0,0,0,...,0,0.0,0,0.0,0,0.000000,0,0.0,0,0.0
4,100016,6,-677.833333,-4067,-1634,-128,6,0.0,0,0,...,0,0.0,0,0.0,0,0.000000,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117030,456236,1,-2319.000000,-2319,-2319,-2319,1,0.0,0,0,...,0,0.0,0,0.0,0,0.000000,0,0.0,0,0.0
117031,456241,1,-2180.000000,-2180,-2180,-2180,1,0.0,0,0,...,0,0.0,0,0.0,0,0.000000,0,0.0,0,0.0
117032,456243,5,-842.800000,-4214,-2358,-154,5,0.0,0,0,...,0,0.0,0,0.0,0,0.000000,0,0.0,0,0.0
117033,456247,7,-874.285714,-6120,-2028,-287,7,0.0,0,0,...,0,0.0,0,0.0,1,0.142857,0,0.0,0,0.0


### The Home Credit tables

Load the Home Credit tables.

In [11]:
credit_card_balance_df = pd.read_csv(os.path.join(dataset_dir, "credit_card_balance.csv")).iloc[:NUM_ROWS]
installments_payments_df = pd.read_csv(os.path.join(dataset_dir, "installments_payments.csv")).iloc[:NUM_ROWS]
POS_CASH_balance_df = pd.read_csv(os.path.join(dataset_dir, "POS_CASH_balance.csv")).iloc[:NUM_ROWS]
previous_application_df = pd.read_csv(os.path.join(dataset_dir, "previous_application.csv")).iloc[:NUM_ROWS]

Aggregate the *POS_CASH_balance* table and merge it with the *previous_application* table

In [12]:
agg_POS_CASH_balance_df = agg_columns(
    POS_CASH_balance_df,
    groupby_columns=["SK_ID_PREV"],
    exclude_columns=["SK_ID_CURR"],
    prefix="POS"
)
merged_previous_application_df = pd.merge(previous_application_df, agg_POS_CASH_balance_df, on="SK_ID_PREV", how="left")

Free memory.

In [13]:
del POS_CASH_balance_df, previous_application_df, agg_POS_CASH_balance_df
gc.collect()

0

Aggregate the *installments_payment* table and merge it with the *previous_application* table

In [14]:
agg_installments_payments_df = agg_columns(
    installments_payments_df,
    groupby_columns=["SK_ID_PREV"],
    exclude_columns=["SK_ID_CURR"],
    prefix="INSTALL"
)
merged_previous_application_df = pd.merge(merged_previous_application_df, agg_installments_payments_df, on="SK_ID_PREV", how="left")

Free memory.

In [15]:
del installments_payments_df, agg_installments_payments_df
gc.collect()

0

Aggregate the *credit_card_balance* table and merge it with the *previous_application* table

In [16]:
agg_credit_card_balance_df = agg_columns(
    credit_card_balance_df,
    groupby_columns=["SK_ID_PREV"],
    exclude_columns=["SK_ID_CURR"],
    prefix="CC"
)
merged_previous_application_df = pd.merge(merged_previous_application_df, agg_credit_card_balance_df, on="SK_ID_PREV", how="left")

Free memory.

In [17]:
del credit_card_balance_df, agg_credit_card_balance_df
gc.collect()

0

Aggregate the merged *previous_application* table.

In [18]:
merged_home_credit_df = agg_columns(
    merged_previous_application_df,
    groupby_columns=["SK_ID_CURR"],
    exclude_columns=["SK_ID_PREV"],
    prefix="PREV"
)

Free memory.

In [19]:
del merged_previous_application_df
gc.collect()

0

Merge the aggregated bureau the Home Credit dataframes.

In [20]:
to_merge_df = pd.merge(merged_bureau_df, merged_home_credit_df, on="SK_ID_CURR", how="outer")

Free memory.

In [21]:
del merged_bureau_df, merged_home_credit_df
gc.collect()

0

### Time features

The features above aggregate each customer's history as a whole, without regard for when something happened. Here we add features that capture how the repayment behaviour evolves over time

In [22]:
installments_payments_df = pd.read_csv(os.path.join(dataset_dir, "installments_payments.csv")).iloc[:NUM_ROWS]
POS_CASH_balance_df = pd.read_csv(os.path.join(dataset_dir, "POS_CASH_balance.csv")).iloc[:NUM_ROWS]

From the *installments_payments* table, for each customer we extract:
- `LAST_MONTH_LATENESS` - the number of days the most recent installment was paid late (0 if paid on time or early),
- `FREQ_1YR` and `FREQ_6MO` - the fraction of installments paid late in the last year / 6 months,
- `SEV_1YR` and `SEV_6MO` - the average number of days late in the last year / 6 months (early payments count as 0),
- `FREQ_ACCELERATION_RATIO` and `SEV_ACCELERATION_RATIO` - the 6-month value relative to the 1-year value; values above 1 mean the payment discipline has recently been getting worse.

In [23]:
def build_time_series_features(df):
    """
    Extracts time-aware frequency, severity and trend metrics from the installments table.
    """
    # Sort by customer and due date so that "last" rows are the most recent ones
    df = df.sort_values(by=["SK_ID_CURR", "DAYS_INSTALMENT"])

    # Days late (positive = late, negative = early); early payments are clipped
    # to 0 so they do not skew the severity
    df["DAYS_LATE"] = df["DAYS_ENTRY_PAYMENT"] - df["DAYS_INSTALMENT"]
    df["DAYS_LATE_CLIPPED"] = df["DAYS_LATE"].clip(lower=0)
    df["IS_LATE"] = (df["DAYS_LATE"] > 0).astype(int)

    # Keep only installments due before the current application
    df = df[df["DAYS_INSTALMENT"] < 0].copy()

    # Lateness of the most recent installment
    last_inst = df.drop_duplicates(subset=["SK_ID_CURR"], keep="last")[["SK_ID_CURR", "DAYS_LATE_CLIPPED"]]
    last_inst = last_inst.rename(columns={"DAYS_LATE_CLIPPED": "LAST_MONTH_LATENESS"})

    # Time windows: the past year and the past 6 months
    mask_1yr = (df["DAYS_INSTALMENT"] >= -365) & (df["DAYS_INSTALMENT"] < 0)
    mask_6mo = (df["DAYS_INSTALMENT"] >= -180) & (df["DAYS_INSTALMENT"] < 0)

    agg_1yr = df[mask_1yr].groupby("SK_ID_CURR").agg(
        FREQ_1YR=("IS_LATE", "mean"),          # fraction of payments late in the last year
        SEV_1YR=("DAYS_LATE_CLIPPED", "mean")  # average days late in the last year
    ).reset_index()

    agg_6mo = df[mask_6mo].groupby("SK_ID_CURR").agg(
        FREQ_6MO=("IS_LATE", "mean"),          # fraction of payments late in the last 6 months
        SEV_6MO=("DAYS_LATE_CLIPPED", "mean")  # average days late in the last 6 months
    ).reset_index()

    # Merge everything onto the list of unique customers
    res = pd.DataFrame({"SK_ID_CURR": df["SK_ID_CURR"].unique()})
    res = res.merge(last_inst, on="SK_ID_CURR", how="left")
    res = res.merge(agg_1yr, on="SK_ID_CURR", how="left")
    res = res.merge(agg_6mo, on="SK_ID_CURR", how="left")

    # Customers with no installments inside a window get 0
    res = res.fillna(0)

    # Trend ratios: > 1 means the recent behaviour is worse than the yearly baseline
    res["FREQ_ACCELERATION_RATIO"] = res["FREQ_6MO"] / (res["FREQ_1YR"] + 0.0001)
    res["SEV_ACCELERATION_RATIO"] = res["SEV_6MO"] / (res["SEV_1YR"] + 0.0001)

    return res

From the *POS_CASH_balance* table (monthly snapshots of previous POS/cash loans), for each customer we extract:
- `POS_TOTAL_STALLED_MONTHS` - the total number of months in which an active loan did not progress (the number of remaining installments stayed the same month-over-month), i.e. missed payment cycles,
- `POS_MAX_DAYS_LATE_LAST_3M` - the worst days-past-due (`SK_DPD`) over the last 3 months,
- `POS_ACTIVE_LOANS_COUNT_LAST_MONTH` - the number of distinct active POS loans in the most recent month.

In [24]:
def build_advanced_pos_features(df):
    """
    Extracts monthly-progression metrics from the POS_CASH_balance table.
    """
    # Sort by customer, loan and time so we can follow the sequence of months of each loan
    df = df.sort_values(by=["SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE"])

    assert (df["MONTHS_BALANCE"] < 0).all(), "Found MONTHS_BALANCE >= 0 - future data leaking in!"

    # Month-over-month change of the remaining installments; a regular payment gives -1
    df["TERM_DIFF"] = df.groupby(["SK_ID_CURR", "SK_ID_PREV"])["CNT_INSTALMENT_FUTURE"].diff()

    # If the difference is 0 and the contract is active, a payment cycle was missed
    df["IS_STALLED_MONTH"] = ((df["TERM_DIFF"] == 0) & (df["NAME_CONTRACT_STATUS"] == "Active")).astype(int)

    customer_features = pd.DataFrame({"SK_ID_CURR": df["SK_ID_CURR"].unique()})

    # Total lifetime stalled payment months
    stalled_counts = df.groupby("SK_ID_CURR")["IS_STALLED_MONTH"].sum().reset_index()
    stalled_counts = stalled_counts.rename(columns={"IS_STALLED_MONTH": "POS_TOTAL_STALLED_MONTHS"})

    # Worst days-past-due in the last 3 months
    mask_recent = df["MONTHS_BALANCE"] >= -3
    recent_dpd = df[mask_recent].groupby("SK_ID_CURR")["SK_DPD"].max().reset_index()
    recent_dpd = recent_dpd.rename(columns={"SK_DPD": "POS_MAX_DAYS_LATE_LAST_3M"})

    # Number of distinct active loans in the most recent month
    mask_last_month = df["MONTHS_BALANCE"] == -1
    active_loans_last_month = (
        df[mask_last_month & (df["NAME_CONTRACT_STATUS"] == "Active")]
        .groupby("SK_ID_CURR")["SK_ID_PREV"]
        .nunique()
        .reset_index()
    )
    active_loans_last_month = active_loans_last_month.rename(
        columns={"SK_ID_PREV": "POS_ACTIVE_LOANS_COUNT_LAST_MONTH"}
    )

    customer_features = customer_features.merge(stalled_counts, on="SK_ID_CURR", how="left")
    customer_features = customer_features.merge(recent_dpd, on="SK_ID_CURR", how="left")
    customer_features = customer_features.merge(active_loans_last_month, on="SK_ID_CURR", how="left")

    # Customers with e.g. no active loans in the last month get 0
    customer_features = customer_features.fillna(0)

    return customer_features

Build the time features and merge them with the aggregated data table. The features are aggregated per customer (`SK_ID_CURR`), so they are merged the same way as the bureau and Home Credit aggregates - before the train/test split, which is safe because every feature is computed only from data preceding the customer's application.

In [25]:
time_series_features_df = build_time_series_features(installments_payments_df)
advanced_pos_features_df = build_advanced_pos_features(POS_CASH_balance_df)

time_features_df = pd.merge(time_series_features_df, advanced_pos_features_df, on="SK_ID_CURR", how="outer")
to_merge_df = pd.merge(to_merge_df, time_features_df, on="SK_ID_CURR", how="outer")

In [28]:
ts_features = time_features_df.columns.tolist()

Free memory.

In [29]:
del installments_payments_df, POS_CASH_balance_df, time_series_features_df, advanced_pos_features_df, time_features_df
gc.collect()

0

## Data preprocessing

Load the main *application_train.csv* table and merge it with the aggregated data table.

In [30]:
applications_df = pd.read_csv(os.path.join(dataset_dir, "application_train.csv")).iloc[:NUM_ROWS]

The `DAYS_EMPLOYED` column uses the value `365243` (which is aroung 1000 years) to mark applicants that are not employed. Replace it with `NaN` so it does not distort the feature scale

In [31]:
applications_df["DAYS_EMPLOYED"] = applications_df["DAYS_EMPLOYED"].replace(365243, np.nan)

In [32]:
from sklearn.model_selection import train_test_split

applications_train, applications_test = train_test_split(
    applications_df,
    train_size=0.7,
    random_state=42,
    stratify=applications_df["TARGET"]
)

applications_train = applications_train.reset_index(drop=True)
applications_test = applications_test.reset_index(drop=True)

Free memory.

In [33]:
del applications_df
gc.collect()

16

Process the training set.

In [34]:
applications_train = pd.merge(applications_train, to_merge_df, on="SK_ID_CURR", how="left")
applications_train = pd.get_dummies(applications_train, drop_first=True)
applications_train = remove_nan_columns(applications_train, threshold=0.5)

We remove the ID column since it does not have any valuable information

In [35]:
applications_train = applications_train.drop(columns=["SK_ID_CURR"])

In [36]:
applications_train

,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,FONDKAPREMONT_MODE_reg oper spec account,HOUSETYPE_MODE_specific housing,HOUSETYPE_MODE_terraced house,WALLSMATERIAL_MODE_Mixed,WALLSMATERIAL_MODE_Monolithic,WALLSMATERIAL_MODE_Others,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_Yes
0,0,2,405000.0,1971072.0,68643.0,1800000.0,0.006852,-13587,-1028.0,-7460.0,...,False,False,False,False,False,False,False,True,False,False
1,0,0,337500.0,508495.5,38146.5,454500.0,0.010276,-17543,-1208.0,-4054.0,...,False,False,False,False,False,False,False,False,False,False
2,0,1,112500.0,110146.5,13068.0,90000.0,0.005084,-11557,-593.0,-5554.0,...,False,False,False,False,False,False,False,False,False,False
3,0,2,40500.0,66384.0,3519.0,45000.0,0.031329,-15750,-5376.0,-5285.0,...,False,False,False,False,False,False,False,False,False,False
4,0,0,225000.0,298512.0,31801.5,270000.0,0.019101,-19912,-1195.0,-86.0,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215252,0,1,157500.0,846517.5,33700.5,684000.0,0.035792,-14742,-7799.0,-5732.0,...,False,False,False,False,False,False,False,False,False,False
215253,0,1,135000.0,405000.0,20250.0,405000.0,0.035792,-15374,-595.0,-6831.0,...,False,False,False,False,False,False,False,False,False,False
215254,0,0,157500.0,272520.0,21528.0,225000.0,0.018801,-19035,-4334.0,-8490.0,...,False,False,False,False,False,False,True,False,False,False
215255,0,0,90000.0,246357.0,24493.5,234000.0,0.025164,-23088,NaN,-8975.0,...,False,False,False,True,False,False,False,False,False,False


Process the test set.

In [39]:
applications_test = pd.merge(applications_test, to_merge_df, on="SK_ID_CURR", how="left")
applications_test = pd.get_dummies(applications_test, drop_first=True)

Align test features to match the features from the training set.

In [40]:
applications_test = applications_test.reindex(columns=applications_train.columns, fill_value=0)

In [41]:
applications_test

,TARGET,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,...,FONDKAPREMONT_MODE_reg oper spec account,HOUSETYPE_MODE_specific housing,HOUSETYPE_MODE_terraced house,WALLSMATERIAL_MODE_Mixed,WALLSMATERIAL_MODE_Monolithic,WALLSMATERIAL_MODE_Others,WALLSMATERIAL_MODE_Panel,"WALLSMATERIAL_MODE_Stone, brick",WALLSMATERIAL_MODE_Wooden,EMERGENCYSTATE_MODE_Yes
0,0,0,90000.0,803259.0,23616.0,670500.0,0.008625,-19596,-1873.0,-10088.0,...,False,False,False,False,False,False,False,False,False,False
1,0,0,270000.0,1288350.0,41692.5,1125000.0,0.008019,-13853,-1070.0,-670.0,...,False,False,False,False,False,False,True,False,False,False
2,0,0,270000.0,253737.0,25227.0,229500.0,0.020246,-20895,-1477.0,-5773.0,...,False,False,False,False,False,False,True,False,False,False
3,0,0,144000.0,436032.0,16564.5,360000.0,0.018850,-18859,-916.0,-5972.0,...,False,False,False,False,False,False,False,True,False,False
4,0,2,72000.0,225000.0,11250.0,225000.0,0.019101,-16521,-2786.0,-6337.0,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92249,0,0,135000.0,450000.0,21780.0,450000.0,0.018029,-17288,-2950.0,-1074.0,...,False,False,False,False,False,False,False,False,False,False
92250,0,0,112500.0,298512.0,23715.0,270000.0,0.006671,-10511,-3692.0,-4144.0,...,False,False,False,False,False,False,False,True,False,False
92251,0,0,157500.0,288873.0,19435.5,238500.0,0.035792,-18694,-910.0,-478.0,...,False,False,False,False,False,False,False,True,False,False
92252,0,0,180000.0,940500.0,39978.0,940500.0,0.026392,-19972,-12748.0,-12277.0,...,False,False,False,False,False,False,False,True,False,False


In [42]:
from sklearn.impute import SimpleImputer

In [43]:
imp_mean = SimpleImputer(strategy="mean").set_output(transform="pandas")
imp_mean.fit(applications_train)
applications_train = imp_mean.transform(applications_train)
applications_test = imp_mean.transform(applications_test)

Save the preprocessed data to disk.

In [44]:
applications_train.to_csv(os.path.join(dataset_dir, "train_set.csv"), index=False)
applications_test.to_csv(os.path.join(dataset_dir, "test_set.csv"), index=False)